# 2do Workshop Eje de Innovación — Retrieval-Augmented Generation (RAG)

**Red Feminista en IA para Latinoamérica y el Caribe**

---

### Pre-requisitos
- Python >= 3.10
- `jupyterlab` (o similar) instalado para ejecutar este notebook

**Todo lo demás se instala automáticamente en la primera celda de código.**

### Modelos del workshop (toda la familia Qwen: un solo ecosistema coherente, multilingüe y abierto)
- **LLM default:** `Qwen/Qwen3-1.7B` — **Fallback (menos recursos):** `Qwen/Qwen3-0.6B`
- **Embeddings:** `Qwen/Qwen3-Embedding-0.6B`
- **Reranker:** `Qwen/Qwen3-Reranker-0.6B`

### Stack
- **Hugging Face** → modelos y datasets
- **LlamaIndex** → pipeline RAG e índice in-memory
- **Qdrant** → base de datos vectorial (modo embebido, sin servidor ni Docker)

In [ ]:
import sys
print(f"Versión de Python en uso: {sys.version}")

## 0. Pathways según nivel y hardware

### 0.1 Objetivo final
Construir un **pipeline RAG sobre documentos propios** de cada proyecto, con respuestas citadas y evaluación básica de calidad.

### 0.2 Elegí tu camino

| Pathway | Secciones | Descripción |
|---------|-----------|-------------|
| 🟢 Exploración simple | 1 → 2 → 3 → 6 → 8 → 12 | Setup → intuición RAG → embeddings → índice in-memory → pipeline RAG → aplicación |
| 🟡 Experimentación | 1 → 2 → 3 → 4 → 5 → 6 → 8 → 10 → 12 | + similaridad → chunking → citaciones |
| 🔴 Profundidad | 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 | Todo el flujo: + Qdrant → reranking → evaluación |
| ⚙️ Hardware limitado | 1 → 2 → 3 → 6 → 8 → 10 → 12 | Mismo flujo con Qwen3-0.6B; el retrieval es liviano, el cuello de botella es el LLM |

### 0.3 Elegí según tu hardware
- 🖥️ **Solo CPU** → Embeddings: `Qwen3-Embedding-0.6B` corre en CPU (indexar toma más tiempo, pero el corpus es chico). LLM: `Qwen3-0.6B`. Qdrant embebido o índice in-memory.
- ⚡ **GPU disponible** → Flujo completo con `Qwen3-1.7B` + reranker (sección 9).
- ❓ **No sabés qué hardware tenés** → Avanzá hasta **1.4 (detección de hardware)** y decidí ahí.

### 0.4 Cómo usar este notebook
- ▶️ **Ejecutar esta celda** → base funcional inmediata
- 🧪 **Probá esto** → modificar y experimentar
- 🚀 **Ir más allá** → profundización opcional

---
## 1. Setup local (adaptado a hardware)

### 1.1 Entorno
Recomendamos ejecutar este notebook dentro de un entorno virtual (`venv` o `conda`) para no mezclar dependencias con otros proyectos:

```bash
python -m venv .venv
source .venv/bin/activate      # Linux / macOS
pip install jupyterlab
jupyter lab
```

### 1.2 Instalación base
La siguiente celda (▶️) instala todo lo que necesitamos:

| Paquete | Rol en el pipeline |
|---------|--------------------|
| `transformers`, `sentence-transformers` | cargar los modelos de Hugging Face (LLM, embeddings, reranker) |
| `llama-index-core` + integraciones | orquestar el pipeline RAG (documentos, índice, query engine) |
| `qdrant-client` | base de datos vectorial embebida (sección 7) — se instala con pip y listo, sin servidor ni Docker |
| `plotly`, `scikit-learn` | visualizaciones interactivas y proyecciones 2D |

In [ ]:
# ═══ 1.2 Instalacion base ═══
import subprocess
import sys

# Asegurar que pip este disponible
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "--version"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
except subprocess.CalledProcessError:
    print("⏳ Instalando pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--default-pip"])

paquetes = [
    "torch",
    "transformers",
    "sentence-transformers",
    "accelerate",
    "huggingface_hub",
    "llama-index-core",
    "llama-index-embeddings-huggingface",
    "llama-index-llms-huggingface",
    "llama-index-vector-stores-qdrant",
    "qdrant-client",
    "numpy",
    "scikit-learn",
    "pandas",
    "plotly",
]

print("📦 Instalando paquetes necesarios (puede tardar unos minutos la primera vez)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + paquetes)
print("✅ Paquetes instalados correctamente")

In [ ]:
# ═══ Imports y configuracion general ═══
import gc
import json
import os
import re
import time
import warnings

import numpy as np
import torch

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ═══ Modelos del workshop ═══
NOMBRE_LLM_DEFAULT = "Qwen/Qwen3-1.7B"        # requiere GPU o >=16GB RAM
NOMBRE_LLM_FALLBACK = "Qwen/Qwen3-0.6B"       # anda bien en CPU / hardware limitado
NOMBRE_EMBEDDINGS = "Qwen/Qwen3-Embedding-0.6B"
NOMBRE_RERANKER = "Qwen/Qwen3-Reranker-0.6B"

# ═══ Paleta para las visualizaciones del taller ═══
COLORES = ["#2a78d6", "#eb6834", "#1baf7a"]
ESCALA_SECUENCIAL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
COLOR_TINTA = "#0b0b0b"
COLOR_GRILLA = "#e1e0d9"

import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

def estilizar_figura(fig, titulo=None):
    """Aplica el estilo comun del taller a una figura de plotly."""
    fig.update_layout(
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color=COLOR_TINTA, size=13),
        paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
        title=titulo, margin=dict(l=60, r=30, t=60 if titulo else 30, b=60),
    )
    fig.update_xaxes(gridcolor=COLOR_GRILLA, zerolinecolor=COLOR_GRILLA)
    fig.update_yaxes(gridcolor=COLOR_GRILLA, zerolinecolor=COLOR_GRILLA)
    return fig

print("✅ Configuración lista")

### 1.3 Qdrant embebido

Qdrant es la base de datos vectorial que vamos a usar en la sección 7. La buena noticia: tiene un **modo local embebido** que guarda todo en un archivo en disco. No hace falta levantar un servidor ni instalar Docker — `pip install qdrant-client` (que ya hicimos arriba) es todo el setup.

### 1.4 Detección de hardware
La siguiente celda detecta tu hardware y elige el LLM apropiado. El **retrieval** (embeddings + búsqueda) es liviano y corre bien en cualquier máquina; el que define qué tan cómodo va a ser el taller es el **LLM generador**.

In [ ]:
# ═══ 1.4 Deteccion de hardware ═══
import platform

print("=" * 55)
print("🔍 DETECCION DE HARDWARE")
print("=" * 55)

print(f"\n💻 Sistema: {platform.system()} {platform.machine()}")
print(f"   Python: {sys.version.split()[0]}")

# RAM
if platform.system() == "Darwin":
    ram_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
elif platform.system() == "Linux":
    ram_bytes = 0
    with open("/proc/meminfo") as f:
        for linea in f:
            if "MemTotal" in linea:
                ram_bytes = int(linea.split()[1]) * 1024
                break
else:
    ram_bytes = 0

ram_gb = ram_bytes / (1024**3) if ram_bytes else 0
if ram_gb > 0:
    print(f"   RAM total: {ram_gb:.1f} GB")

# GPU / MPS
hay_cuda = torch.cuda.is_available()
hay_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

if hay_cuda:
    nombre_gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    dispositivo = "cuda"
    print(f"\n⚡ GPU detectada: {nombre_gpu} ({vram_gb:.1f} GB VRAM)")
elif hay_mps:
    dispositivo = "mps"
    vram_gb = ram_gb  # memoria unificada
    print("\n⚡ GPU detectada: 🍎 Apple Silicon (MPS backend)")
else:
    dispositivo = "cpu"
    vram_gb = 0
    print("\n🖥️ Solo CPU disponible")

# Eleccion automatica del LLM (podes sobreescribirla a mano)
if (hay_cuda and vram_gb >= 6) or ram_gb >= 16:
    nombre_llm = NOMBRE_LLM_DEFAULT
else:
    nombre_llm = NOMBRE_LLM_FALLBACK

print("\n" + "=" * 55)
print(f"✅ Dispositivo elegido:  {dispositivo}")
print(f"✅ LLM elegido:          {nombre_llm}")
print(f"   Embeddings:           {NOMBRE_EMBEDDINGS}")
print(f"   Reranker (sección 9): {NOMBRE_RERANKER}")
print("=" * 55)

# 🧪 Proba esto: forzar el otro modelo descomentando la linea que quieras
# nombre_llm = NOMBRE_LLM_DEFAULT
# nombre_llm = NOMBRE_LLM_FALLBACK

### 1.5 Corpus del workshop

Para que todas las personas trabajemos sobre los mismos documentos, la siguiente celda crea la carpeta `corpus_taller/` con los documentos de una **biblioteca comunitaria ficticia** (la *Biblioteca Comunitaria Arroyo Claro*), en español y portugués: reglamentos, actas, guías y FAQs.

¿Por qué un corpus inventado? Porque así podemos verificar con certeza qué información **está** y qué información **no está** en los documentos — algo clave cuando evaluemos alucinaciones (sección 10). Además, como es ficticio, el modelo no puede "acordarse" de nada: **todo lo que responda bien tiene que salir del retrieval**.

> 🧪 **Probá esto (para tu proyecto):** creá una carpeta `mis_documentos/` con 5–10 documentos propios (`.txt` o `.md`) y usala en la sección 12.

In [ ]:
# ═══ 1.5 Crear el corpus del workshop ═══
from pathlib import Path

CARPETA_CORPUS = Path("corpus_taller")
CARPETA_CORPUS.mkdir(exist_ok=True)

documentos_taller = {
    "historia_biblioteca.md": """\
# Historia de la Biblioteca Comunitaria Arroyo Claro

La Biblioteca Comunitaria Arroyo Claro fue fundada en 2011 por un grupo de
vecinas y vecinos del barrio Arroyo Claro. Comenzó funcionando en un galpón
prestado, con una colección inicial de 300 libros donados.

En 2015 la biblioteca se mudó a su local actual, ubicado junto a la estación
de tren. Gracias a campañas de donación y a un convenio con editoriales
independientes, la colección creció hasta superar los 12.000 ejemplares.

La biblioteca es gestionada por una comisión de 7 personas, elegidas en
asamblea cada 2 años. Todas las actividades son organizadas por personas
voluntarias y el acceso a la biblioteca es libre y gratuito.
""",
    "reglamento_prestamos.md": """\
# Reglamento de préstamos

1. Pueden retirar materiales todas las personas asociadas a la biblioteca.
   Asociarse es gratuito: solo hace falta completar un formulario con datos
   de contacto.
2. Cada persona puede llevarse hasta 3 libros a la vez, por un plazo de
   21 días.
3. El préstamo puede renovarse una única vez, por 14 días más, siempre que
   el libro no haya sido reservado por otra persona. La renovación puede
   hacerse en el mostrador o por correo electrónico.
4. Los materiales de referencia (enciclopedias, diccionarios y mapas) no se
   prestan a domicilio: se consultan únicamente en la sala de lectura.
5. En caso de atraso en la devolución, se suspende la posibilidad de retirar
   nuevos materiales por el doble de días del atraso.
6. La cuota social es voluntaria y se destina al mantenimiento del edificio
   y a la compra de nuevos materiales.
""",
    "talleres_y_actividades.md": """\
# Talleres y actividades regulares

- **Taller de huerta**: sábados a las 10:00, en el patio trasero. Abierto a
  todas las edades. Coordinado por el grupo de la huerta comunitaria.
- **Club de lectura**: primer jueves de cada mes a las 18:00, en la sala de
  lectura. Cada mes se vota el libro siguiente.
- **Taller de ajedrez**: miércoles a las 17:00. A partir de los 8 años.
  Hay tableros disponibles en la biblioteca, no hace falta traer.
- **Apoyo escolar**: lunes y viernes de 16:00 a 18:00, para estudiantes de
  primaria y secundaria.

La inscripción a todos los talleres es gratuita y puede hacerse en el
mostrador de la biblioteca o escribiendo por correo electrónico.
""",
    "guia_huerta_semillas.md": """\
# Guía del banco de semillas

La biblioteca mantiene un banco de semillas comunitario desde 2018, junto
al grupo de la huerta.

## ¿Cómo funciona el intercambio?

Cualquier persona puede llevarse hasta 3 sobres de semillas por temporada.
El compromiso es simple: después de la cosecha, devolver al banco el doble
de las semillas retiradas, para que el banco siga creciendo.

## Variedades disponibles

- Tomate platense
- Zapallo anco
- Albahaca de hoja ancha
- Acelga

## Calendario breve de siembra

- **Primavera**: tomate, albahaca y zapallo.
- **Otoño**: acelga y otras verduras de hoja.

Las semillas se entregan en sobres de papel etiquetados con la variedad y
la fecha de cosecha.
""",
    "acervo_emprestimos_pt.md": """\
# Acervo em português

A biblioteca possui uma seção com cerca de 800 títulos em português,
principalmente de literatura brasileira contemporânea, formada a partir de
um intercâmbio com uma biblioteca comunitária de Porto Alegre iniciado
em 2019.

As regras de empréstimo são as mesmas do regulamento geral: até 3 livros
por vez, por 21 dias.

Doações de livros em português são recebidas às sextas-feiras, das 14:00
às 18:00, diretamente no balcão. Antes de doar coleções grandes (mais de
50 livros), pedimos que entre em contato por correio eletrônico para
coordenar a entrega.
""",
    "oficina_compostagem_pt.md": """\
# Oficina de compostagem

A oficina de compostagem acontece na última terça-feira de cada mês, às
17:00, no pátio da biblioteca. A participação é gratuita e aberta a todas
as pessoas.

A biblioteca mantém três composteiras no pátio, que recebem os restos
orgânicos da copa e das atividades.

## O que pode ir na composteira

Cascas de frutas e legumes, borra de café, folhas secas, papel picado
sem tinta colorida.

## O que não pode ir na composteira

Carnes, laticínios, gorduras, frutas cítricas em grande quantidade e
qualquer material plástico.

O composto produzido é usado na horta comunitária e distribuído entre as
pessoas participantes da oficina.
""",
    "equipamiento_sala_digital.md": """\
# Sala digital: equipamiento y uso

La sala digital de la biblioteca cuenta con:

- 6 computadoras de escritorio con acceso a internet.
- 1 impresora 3D, donada en 2022 por una cooperativa tecnológica del barrio.
- Conexión wifi abierta, red "BiblioArroyo", disponible en todo el edificio.

## Uso de la impresora 3D

La impresora 3D puede reservarse con al menos 48 horas de anticipación en
el mostrador. Cada persona puede reservar hasta 4 horas de impresión por
semana. El material de impresión (filamento PLA) lo aporta la biblioteca
para proyectos comunitarios y educativos.

## Cursos

Los martes a las 18:00 se dicta el curso de alfabetización digital, de
nivel inicial, sin requisitos previos.
""",
    "acta_asamblea_2024.md": """\
# Acta de la asamblea general — marzo de 2024

Resumen de las decisiones tomadas por la asamblea general de personas
asociadas, realizada en marzo de 2024:

1. **Nuevo horario de atención**: la biblioteca abrirá de lunes a sábado,
   de 9:00 a 19:00. Antes cerraba a las 17:00.
2. **Fondo de reparación de techos**: se crea un fondo específico para
   reparar los techos de la sala de lectura, con lo recaudado en la feria
   anual del libro usado.
3. **Ampliación de la comisión**: se incorporan 2 personas a la comisión
   directiva, que pasa de 7 a 9 integrantes hasta la próxima elección.
4. **Próxima asamblea**: se convocará en marzo de 2025.
""",
}

for nombre_archivo, contenido in documentos_taller.items():
    (CARPETA_CORPUS / nombre_archivo).write_text(contenido, encoding="utf-8")

print(f"✅ Corpus creado en '{CARPETA_CORPUS}/' con {len(documentos_taller)} documentos:")
for nombre_archivo in documentos_taller:
    tamanio = len((CARPETA_CORPUS / nombre_archivo).read_text(encoding="utf-8"))
    idioma = "PT" if nombre_archivo.endswith("_pt.md") else "ES"
    print(f"   [{idioma}] {nombre_archivo:<32} ({tamanio} caracteres)")

---
## 2. RAG: contexto mínimo

### 2.1 El problema: lo que el modelo no sabe

Un LLM tiene el conocimiento **congelado en el momento de su entrenamiento**. Por diseño, no sabe nada de:

- **Datos privados o locales**: los documentos internos de tu organización, actas, entrevistas, bases de conocimiento comunitarias.
- **Datos recientes**: cualquier cosa posterior a su fecha de corte de entrenamiento.
- **Datos demasiado específicos**: aunque hayan estado en internet, detalles de nicho pueden no haber quedado "grabados".

Y lo más problemático: cuando le preguntamos algo que no sabe, muchas veces **no dice "no sé"** — inventa una respuesta plausible. Eso es una **alucinación** (lo vimos en el workshop anterior, sección 8.5).

Nuestro corpus de la biblioteca Arroyo Claro es ficticio, así que es un caso extremo y perfecto: el modelo no puede saber *nada* de él. Cualquier respuesta correcta va a tener que salir de los documentos.

### 2.2 La idea central de RAG

**Buscar primero, generar después.**

En lugar de pedirle al LLM que responda "de memoria", el pipeline RAG:

1. **Busca** en nuestros documentos los fragmentos más relevantes para la pregunta.
2. **Arma un prompt** que incluye esos fragmentos como contexto.
3. **Genera** la respuesta usando ese contexto recuperado, no solo su memoria.

El LLM pasa de ser un "oráculo que sabe todo" a ser un **redactor que lee y sintetiza** lo que le acercamos.

### 2.3 Anatomía de un pipeline RAG

Este es el diagrama de referencia que vamos a construir **pieza por pieza** en el resto del notebook:

```
  INDEXACIÓN (una sola vez, offline)
  ┌──────────────┐   ┌──────────┐   ┌────────────┐   ┌──────────────────┐
  │  documentos  │ → │ chunking │ → │ embeddings │ → │  índice vectorial│
  │  (sección 1) │   │ (secc 5) │   │  (secc 3)  │   │  (secc 6 y 7)    │
  └──────────────┘   └──────────┘   └────────────┘   └──────────────────┘

  CONSULTA (cada vez que alguien pregunta)
  ┌──────────┐   ┌───────────┐   ┌─────────────┐   ┌────────────────┐
  │ pregunta │ → │ retrieval │ → │ (reranking) │ → │  generación    │
  │          │   │ (secc 4-6)│   │  (secc 9)   │   │  con citas     │
  └──────────┘   └───────────┘   └─────────────┘   │  (secc 8 y 10) │
                                                   └────────────────┘
```

### 2.4 RAG vs fine-tuning vs prompting largo

| | **RAG** | **Fine-tuning** | **Prompting largo** (todo en el prompt) |
|---|---|---|---|
| ¿Qué hace? | busca y adjunta contexto relevante | ajusta los pesos del modelo | pega todos los documentos en el prompt |
| Actualizar conocimiento | agregar/editar documentos, sin reentrenar | reentrenar cada vez | editar el prompt |
| Costo | bajo (indexar es barato) | alto (GPU, datos, tiempo) | crece con cada consulta (tokens) |
| Trazabilidad | alta: se puede citar la fuente | nula: el conocimiento queda "diluido" en los pesos | media |
| Límite principal | calidad del retrieval | necesita muchos ejemplos | ventana de contexto del modelo |
| Ideal para | conocimiento factual que cambia | **estilo, formato, tono** | corpus muy chicos (< algunas páginas) |

Regla práctica: **RAG para conocimiento, fine-tuning para comportamiento.** Y se pueden combinar.

### 2.5 Casos de uso aplicados a los proyectos

- **Bases de conocimiento comunitarias**: responder preguntas frecuentes sobre el funcionamiento de una organización, con fuentes citadas.
- **Documentación interna**: reglamentos, manuales, actas de reuniones que nadie tiene tiempo de releer.
- **Archivos y entrevistas**: hacer navegable un archivo de historia oral o de entrevistas transcriptas.
- **Normativas**: encontrar qué dice una ordenanza o un reglamento sobre un tema puntual, con referencia al texto original.

> 💬 **Para discutir en tu equipo:** ¿qué documentos tiene tu proyecto que la gente consulta una y otra vez? Esa es tu primera candidata a corpus RAG.

---
## 3. Embeddings: representar significado con vectores

### 3.1 Qué es un embedding

Un **embedding** convierte un texto en un **vector de números** (una lista de, por ejemplo, 1024 valores). La propiedad mágica: textos con **significado parecido** quedan **cerca en el espacio** vectorial, aunque no compartan ninguna palabra.

```
  "¿A qué hora abre la biblioteca?"   →  [ 0.12, -0.48, 0.33, ... ]   ┐
  "horario de atención al público"    →  [ 0.10, -0.45, 0.30, ... }   ├─ cerca
                                                                      ┘
  "receta de pan casero"              →  [-0.52,  0.07, 0.81, ... ]   ← lejos
```

Esa intuición geométrica — **cerca en el espacio = parecido en significado** — es el corazón de todo el retrieval que viene.

### 3.2 Cargar un modelo de embeddings (Hugging Face)

Vamos a usar `Qwen/Qwen3-Embedding-0.6B` a través de la librería `sentence-transformers`, que nos da una interfaz simple (`.encode()`). La alternativa es usar `transformers` "crudo" y hacer el pooling a mano — más control, más código; para este taller no hace falta.

In [ ]:
# ═══ 3.2 Cargar el modelo de embeddings ═══
from sentence_transformers import SentenceTransformer

print(f"⏳ Cargando modelo de embeddings: {NOMBRE_EMBEDDINGS}")
print("   (la primera vez descarga los pesos, ~1.2 GB)")

modelo_embeddings = SentenceTransformer(NOMBRE_EMBEDDINGS, device=dispositivo)

dimension = modelo_embeddings.get_embedding_dimension()
print(f"✅ Modelo cargado")
print(f"   Dimensión del vector: {dimension}")
print(f"   Parámetros: ~0.6B — chico para ser un LLM, potente para ser un embedder")

### 3.3 Primer ejercicio: embeddear frases

▶️ Convirtamos algunas frases en vectores y miremos qué tienen adentro.

In [ ]:
# ═══ 3.3 Embeddear frases e inspeccionar el vector ═══
frases = [
    "¿Cuántos libros me puedo llevar de la biblioteca?",
    "Reglamento de préstamo de materiales",
    "El taller de huerta es los sábados a la mañana",
    "Receta de pan casero con masa madre",
]

vectores = modelo_embeddings.encode(frases)

print(f"Cantidad de frases: {len(frases)}")
print(f"Shape de la matriz de vectores: {vectores.shape}")
print(f"  → {vectores.shape[0]} frases, cada una representada por {vectores.shape[1]} números\n")

print(f"Primeros 8 valores del vector de la frase 1:")
print(f"  {np.round(vectores[0][:8], 4)}")
print(f"\nNorma (largo) del vector 1: {np.linalg.norm(vectores[0]):.4f}")
print("  → sentence-transformers ya devuelve vectores normalizados (norma ≈ 1)")

# 🧪 Proba esto: cambia las frases por frases de TU proyecto y volve a ejecutar

### 3.4 Embeddings multilingües y equidad

En el workshop anterior vimos que la **tokenización es desigual entre idiomas** (una misma frase "cuesta" más tokens en guaraní que en inglés). Con los embeddings pasa algo análogo: los modelos se entrenan mayormente con inglés (y chino, en el caso de Qwen), y la calidad de la representación **se degrada en lenguas menos representadas**.

La prueba clave: ¿frases **equivalentes** en distintos idiomas caen **cerca** en el espacio vectorial?

In [ ]:
# ═══ 3.4 ¿Frases equivalentes en distintos idiomas caen cerca? ═══
frases_equivalentes = {
    "español":   "El agua está fría",
    "português": "A água está fria",
    "english":   "The water is cold",
    "guaraní":   "Y iro'ysã",
}
frase_control = "La impresora 3D se reserva con 48 horas de anticipación"

textos = list(frases_equivalentes.values()) + [frase_control]
etiquetas = list(frases_equivalentes.keys()) + ["control (otro tema)"]
vecs = modelo_embeddings.encode(textos)

print("Similaridad coseno contra la frase en ESPAÑOL:\n")
for i in range(1, len(textos)):
    sim = float(vecs[0] @ vecs[i])  # vectores normalizados: producto punto = coseno
    barra = "█" * int(sim * 40)
    print(f"  {etiquetas[i]:<20} {sim:.3f}  {barra}")

print("""
📝 Que mirar:
  - portugues e ingles suelen quedar MUY cerca del español (idiomas bien
    representados en el entrenamiento)
  - guarani suele quedar mas lejos, aun siendo la MISMA idea: el modelo
    lo vio poco durante el entrenamiento
  - la frase de control (otro tema, mismo idioma) deberia quedar mas lejos
    que las traducciones... ¿se cumple tambien para guarani?
""")

> 💬 **Para discutir:** si tu corpus está en una lengua o variedad poco representada, el retrieval va a ser menos preciso — y eso es **invisible** si solo probás en español. ¿Cómo lo detectarías en tu proyecto? (Pista: sección 11, evaluación.)

### 3.5 Visualización: el mapa del espacio semántico

Los vectores tienen 1024 dimensiones — imposibles de dibujar. Pero podemos **proyectarlos a 2D** con PCA (análisis de componentes principales) para ver su organización. La proyección pierde información, pero conserva la estructura gruesa: los **clusters temáticos**.

▶️ La visualización es **interactiva**: pasá el mouse por cada punto para ver la frase completa.

In [ ]:
# ═══ 3.5 Proyectar embeddings a 2D e inspeccionar clusters ═══
from sklearn.decomposition import PCA

grupos = {
    "biblioteca": [
        "¿Cuántos libros puedo llevarme en préstamo?",
        "La renovación se pide en el mostrador",
        "Los diccionarios se consultan en sala",
        "El horario de atención es de 9 a 19",
        "Doações de livros são recebidas às sextas-feiras",
    ],
    "huerta": [
        "En primavera se siembran tomates y albahaca",
        "El compost se prepara con restos de verduras",
        "Las semillas se guardan en sobres de papel",
        "La acelga se cosecha en otoño",
        "A composteira não recebe carnes nem laticínios",
    ],
    "tecnología": [
        "La impresora 3D usa filamento PLA",
        "La sala tiene seis computadoras con internet",
        "El wifi de la biblioteca es una red abierta",
        "El curso de alfabetización digital es los martes",
        "A impressora 3D pode ser reservada com antecedência",
    ],
}

todas_frases = [f for fs in grupos.values() for f in fs]
todos_grupos = [g for g, fs in grupos.items() for _ in fs]

vecs_corpus = modelo_embeddings.encode(todas_frases)
proyeccion = PCA(n_components=2).fit_transform(vecs_corpus)

fig = go.Figure()
for i, (nombre_grupo, color) in enumerate(zip(grupos, COLORES)):
    mascara = [g == nombre_grupo for g in todos_grupos]
    fig.add_trace(go.Scatter(
        x=proyeccion[mascara, 0], y=proyeccion[mascara, 1],
        mode="markers", name=nombre_grupo,
        marker=dict(size=11, color=color, line=dict(width=1, color="#fcfcfb")),
        text=[f for f, m in zip(todas_frases, mascara) if m],
        hovertemplate="%{text}<extra>" + nombre_grupo + "</extra>",
    ))

estilizar_figura(fig, "Espacio semántico proyectado a 2D (PCA) — pasá el mouse por los puntos")
fig.update_layout(xaxis_title="componente 1", yaxis_title="componente 2", height=480)
fig.show()

print("📝 Cada punto es una frase; el color es su tema. Fijate que:")
print("   - las frases del mismo tema forman clusters, SIN compartir palabras")
print("   - las frases en portugués caen dentro del cluster de su tema:")
print("     el espacio es (mayormente) multilingüe")

# 🧪 Proba esto: agrega un grupo con 5 frases de tu proyecto y mira donde cae

---
## 4. Similaridad: medir cercanía entre vectores

### 4.1 Similaridad coseno

La métrica estándar para comparar embeddings de texto es la **similaridad coseno**: el coseno del **ángulo** entre dos vectores.

$$\text{sim}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\| \, \|\vec{b}\|}$$

- Mide **dirección**, no magnitud: no importa cuán "largos" son los vectores, sino hacia dónde apuntan.
- Rango: de -1 (opuestos) a 1 (idénticos). En embeddings de texto casi siempre cae entre 0 y 1.

▶️ La implementación son 3 líneas de numpy:

In [ ]:
# ═══ 4.1 Similaridad coseno en 3 lineas ═══
def similaridad_coseno(a, b):
    """Coseno del angulo entre dos vectores."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

v1, v2 = modelo_embeddings.encode(["hola, ¿cómo estás?", "buenas, ¿qué tal?"])
v3 = modelo_embeddings.encode(["el tomate se siembra en primavera"])[0]

print(f"'hola como estas' vs 'buenas que tal':      {similaridad_coseno(v1, v2):.3f}")
print(f"'hola como estas' vs 'el tomate se siembra': {similaridad_coseno(v1, v3):.3f}")

### 4.2 Otras métricas

| Métrica | Fórmula | Cuándo se usa |
|---------|---------|---------------|
| **Coseno** | ángulo entre vectores | default en texto: el "largo" del vector no aporta significado |
| **Producto punto** | $\vec{a} \cdot \vec{b}$ | cuando la magnitud importa (p. ej. popularidad en recomendación) |
| **Distancia euclidiana** | $\|\vec{a} - \vec{b}\|$ | espacios donde la distancia absoluta tiene sentido físico |

Dato clave: si los vectores están **normalizados** (norma = 1, como los nuestros), el coseno y el producto punto **coinciden**, y la distancia euclidiana ordena igual. Por eso las bases vectoriales pueden usar la métrica más barata de calcular.

In [ ]:
# ═══ 4.2 Con vectores normalizados, coseno == producto punto ═══
a, b = modelo_embeddings.encode(["préstamo de libros", "retirar materiales de la biblioteca"])

print(f"Norma de a: {np.linalg.norm(a):.4f}   Norma de b: {np.linalg.norm(b):.4f}")
print(f"Similaridad coseno:  {similaridad_coseno(a, b):.6f}")
print(f"Producto punto:      {np.dot(a, b):.6f}")
print(f"Distancia euclidiana: {np.linalg.norm(a - b):.4f}")
print(f"  → relación exacta: dist² = 2 - 2·coseno → {np.sqrt(2 - 2 * np.dot(a, b)):.4f}")

### 4.3 Ejercicio: matriz de similaridad con pares trampa

Armemos una matriz de similaridad entre frases elegidas con malicia:

- **misma palabra, distinto sentido**: "banco" (para sentarse) vs "banco" (de semillas)
- **distinto idioma, mismo sentido**: préstamo de libros en ES y PT

▶️ La matriz es interactiva: pasá el mouse por cada celda.

In [ ]:
# ═══ 4.3 Matriz de similaridad ═══
frases_trampa = [
    "Me senté en el banco de la plaza",              # banco: mueble
    "El banco de semillas abre los sábados",         # banco: institución/colección
    "Retiré tres libros en préstamo",                # préstamo ES
    "Peguei três livros emprestados",                # préstamo PT (mismo sentido)
    "La biblioteca presta hasta tres libros",        # préstamo ES (parafraseo)
    "El zapallo se cosecha en otoño",                # huerta
]
etiquetas_cortas = ["banco (plaza)", "banco (semillas)", "préstamo ES",
                    "empréstimo PT", "presta libros", "zapallo"]

vecs_trampa = modelo_embeddings.encode(frases_trampa)
matriz = vecs_trampa @ vecs_trampa.T  # normalizados: producto punto = coseno

fig = go.Figure(go.Heatmap(
    z=np.round(matriz, 2),
    x=etiquetas_cortas, y=etiquetas_cortas,
    colorscale=[[i / (len(ESCALA_SECUENCIAL) - 1), col] for i, col in enumerate(ESCALA_SECUENCIAL)],
    zmin=0, zmax=1,
    text=np.round(matriz, 2), texttemplate="%{text}",
    hovertemplate="%{y}<br>vs %{x}<br>similaridad: %{z}<extra></extra>",
    colorbar=dict(title="coseno"),
))
estilizar_figura(fig, "Matriz de similaridad coseno — pares trampa")
fig.update_layout(height=520, yaxis_autorange="reversed")
fig.show()

print("📝 Que mirar:")
print("   - 'préstamo ES' vs 'empréstimo PT' (distinto idioma, mismo sentido): ALTA")
print("   - 'banco (plaza)' vs 'banco (semillas)' (misma palabra, otro sentido):")
print("     más baja que los pares de significado equivalente")
print("   - 'zapallo' queda lejos de todo lo bibliotecario")

### 4.4 Búsqueda semántica mínima (sin frameworks)

Con lo que ya tenemos podemos construir **retrieval completo en ~10 líneas de numpy**: embeddear el corpus, embeddear la pregunta, ordenar por similaridad y quedarnos con el top-k.

Vale la pena entender esta celda **antes** de delegarle el trabajo a LlamaIndex y Qdrant: por debajo, hacen exactamente esto (más rápido y con más orden).

In [ ]:
# ═══ 4.4 RAG-retrieval en 10 lineas ═══
# 1. Corpus: cada parrafo de cada documento es una unidad de busqueda
parrafos, origenes = [], []
for nombre_archivo in sorted(os.listdir(CARPETA_CORPUS)):
    texto = (CARPETA_CORPUS / nombre_archivo).read_text(encoding="utf-8")
    for parrafo in texto.split("\n\n"):
        if parrafo.strip():
            parrafos.append(parrafo.strip())
            origenes.append(nombre_archivo)

# 2. Embeddear TODO el corpus (esto se hace UNA vez y se guarda: es el "indice")
vecs_parrafos = modelo_embeddings.encode(parrafos)
print(f"Índice casero listo: {len(parrafos)} párrafos embeddeados\n")

# 3. Buscar: embeddear la pregunta y ordenar por similaridad
def buscar(pregunta, k=3):
    vec_pregunta = modelo_embeddings.encode([pregunta])[0]
    similaridades = vecs_parrafos @ vec_pregunta
    top_k = np.argsort(similaridades)[::-1][:k]
    return [(similaridades[i], origenes[i], parrafos[i]) for i in top_k]

for sim, origen, parrafo in buscar("¿cuántos libros me puedo llevar?"):
    print(f"[{sim:.3f}] ({origen})")
    print(f"   {parrafo}...\n")

# 🧪 Proba esto: busca en portugues ("quando posso doar livros?") y mira
# si encuentra el documento correcto aunque el indice este mayormente en español

### 4.5 Discusión: similaridad ≠ relevancia

La búsqueda por vecinos más cercanos tiene límites que conviene conocer desde el día 1:

- **Similaridad temática no es relevancia**: para "¿cuántos libros puedo llevarme?", un párrafo que dice "los libros son maravillosos" puede ser muy *similar* y nada *relevante*.
- **Preguntas con negación o condiciones** ("¿qué NO puede ir en la compostera?") se parecen mucho a su opuesto.
- **Información repartida**: si la respuesta requiere combinar dos párrafos lejanos, el top-k puede traer solo uno.

Parte de estas limitaciones se atacan con **reranking** (sección 9) y con buen **chunking** (sección 5, ya mismo).

---
## 5. Chunking: partir documentos en pedazos útiles

### 5.1 Por qué hay que partir los documentos

Dos razones:

1. **Límite de contexto del LLM**: no podemos meter 12.000 libros (ni 12 documentos largos) en un prompt.
2. **Precisión del retrieval**: el embedding de un documento entero "promedia" todos sus temas.
   - Chunks **grandes** → diluyen: el fragmento relevante queda enterrado entre párrafos que no lo son.
   - Chunks **chicos** → descontextualizan: "se prestan por 21 días" sin saber *qué* se presta.

### 5.2 Estrategias

- **Tamaño fijo** (por tokens o caracteres): simple, ignora la estructura.
- **Por oraciones** (`SentenceSplitter` de LlamaIndex): corta intentando respetar límites de oración. Es el default razonable.
- **Por estructura** (títulos, párrafos, markdown): ideal cuando los documentos tienen estructura clara (`MarkdownNodeParser`).

### 5.3 Parámetros clave

- `chunk_size`: tamaño objetivo del chunk (en tokens).
- `chunk_overlap`: cuántos tokens se repiten entre chunks consecutivos, para no cortar ideas al medio.
- **Metadata que viaja con cada chunk**: fuente, título, página. Es lo que después permite **citar** (sección 10) y **filtrar** (sección 7.5).

In [ ]:
# ═══ 5.2/5.3 SentenceSplitter en accion ═══
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document

texto_ejemplo = (CARPETA_CORPUS / "reglamento_prestamos.md").read_text(encoding="utf-8")
doc_ejemplo = Document(text=texto_ejemplo, metadata={"file_name": "reglamento_prestamos.md"})

separador = SentenceSplitter(chunk_size=64, chunk_overlap=10)
chunks = separador.get_nodes_from_documents([doc_ejemplo])

print(f"Documento original: {len(texto_ejemplo)} caracteres")
print(f"Con chunk_size=64 tokens y overlap=10 → {len(chunks)} chunks\n")
print("═" * 60)
for i, chunk in enumerate(chunks[:3]):
    print(f"CHUNK {i} (metadata: {chunk.metadata.get('file_name')})")
    print(chunk.get_content())
    print("═" * 60)
print("(se muestran solo los primeros 3)")

### 5.4 Ejercicio: 3 configuraciones, mismo documento, misma pregunta

Chunkeemos **todo el corpus** con tres configuraciones distintas y comparemos **qué recupera cada una** para la misma pregunta, usando nuestro buscador casero de la sección 4.4.

In [ ]:
# ═══ 5.4 Comparar 3 configuraciones de chunking ═══
from llama_index.core import SimpleDirectoryReader

# Nota: la metadata viaja DENTRO del chunk al embeddearlo, asi que le dejamos
# solo el nombre del archivo (la default del reader ocupa ~32 tokens por chunk)
documentos = SimpleDirectoryReader(
    str(CARPETA_CORPUS),
    file_metadata=lambda ruta: {"file_name": os.path.basename(ruta)},
).load_data()

configuraciones = {
    "chica (64 tokens)": SentenceSplitter(chunk_size=64, chunk_overlap=0),
    "media (128 tokens)": SentenceSplitter(chunk_size=128, chunk_overlap=20),
    "grande (512 tokens)": SentenceSplitter(chunk_size=512, chunk_overlap=50),
}

PREGUNTA_PRUEBA = "¿Por cuántos días se puede renovar un préstamo?"

resultados_config = {}
for nombre_config, splitter in configuraciones.items():
    nodos = splitter.get_nodes_from_documents(documentos)
    textos_nodos = [n.get_content() for n in nodos]
    vecs_nodos = modelo_embeddings.encode(textos_nodos)
    vec_pregunta = modelo_embeddings.encode([PREGUNTA_PRUEBA])[0]
    mejor = int(np.argmax(vecs_nodos @ vec_pregunta))
    resultados_config[nombre_config] = {
        "n_chunks": len(nodos),
        "largo_promedio": np.mean([len(t) for t in textos_nodos]),
        "mejor_chunk": textos_nodos[mejor],
        "score": float((vecs_nodos @ vec_pregunta)[mejor]),
    }

print(f"PREGUNTA: {PREGUNTA_PRUEBA}\n")
for nombre_config, r in resultados_config.items():
    print(f"── {nombre_config}: {r['n_chunks']} chunks (promedio {r['largo_promedio']:.0f} caracteres)")
    print(f"   Mejor chunk [{r['score']:.3f}]:")
    contenido = r["mejor_chunk"]
    print("   " + (contenido.replace(chr(10), chr(10) + "   ")))
    print()

### 5.5 Discusión: no hay chunking universal

La configuración depende del **documento** y de las **preguntas**:

| Tipo de documento | Estrategia sugerida |
|---|---|
| **Normativas / reglamentos** | chunks por artículo o inciso (estructura); overlap bajo |
| **Entrevistas / historia oral** | chunks más grandes (una idea puede tomar varios turnos de habla); overlap alto |
| **FAQs** | un chunk por pregunta-respuesta (estructura perfecta para RAG) |
| **Actas / minutas** | por punto del orden del día |

> 🧪 **Probá esto:** cambiá `PREGUNTA_PRUEBA` por una pregunta cuya respuesta necesite *contexto* (p. ej. "¿qué materiales no salen de la sala de lectura?") y fijate qué configuración gana ahora.

---
## 6. Índice vectorial in-memory con LlamaIndex

### 6.1 De piezas sueltas a framework

Hasta acá construimos todo a mano: leer archivos, chunkear, embeddear, buscar con numpy. **LlamaIndex** empaqueta exactamente ese flujo en tres abstracciones:

- **`Document`**: un archivo/fuente con su metadata.
- **`Node`**: un chunk con metadata y relaciones (lo que generaba nuestro `SentenceSplitter`).
- **`VectorStoreIndex`**: el índice — embeddea los nodes y resuelve la búsqueda top-k.

Lo importante: **ya sabemos qué hace cada pieza por dentro**, así que el framework deja de ser una caja negra.

### 6.2 Construir el índice

Conectamos **nuestro** modelo de embeddings de Hugging Face (nada de APIs pagas) mediante `Settings`, la configuración global de LlamaIndex.

In [ ]:
# ═══ 6.2 Construir el indice in-memory ═══
# Liberamos la copia "cruda" del modelo de embeddings: LlamaIndex carga la suya
# (usamos pop para que la celda funcione aunque hayas salteado secciones)
for _variable in ["modelo_embeddings", "vecs_parrafos", "vecs_corpus", "vecs_trampa"]:
    globals().pop(_variable, None)
gc.collect()

from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print(f"⏳ Conectando embeddings de Hugging Face a LlamaIndex: {NOMBRE_EMBEDDINGS}")
Settings.embed_model = HuggingFaceEmbedding(model_name=NOMBRE_EMBEDDINGS, device=dispositivo)
Settings.llm = None                      # todavia sin LLM: solo retrieval (seccion 8 lo conecta)
Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=30)

# file_metadata: agregamos el idioma como metadata de cada documento
def metadata_archivo(ruta):
    nombre = os.path.basename(ruta)
    return {"file_name": nombre, "idioma": "pt" if nombre.endswith("_pt.md") else "es"}

documentos = SimpleDirectoryReader(str(CARPETA_CORPUS), file_metadata=metadata_archivo).load_data()
print(f"   {len(documentos)} documentos cargados")

t0 = time.time()
indice = VectorStoreIndex.from_documents(documentos, show_progress=True)
print(f"✅ Índice construido en {time.time() - t0:.1f}s")

### 6.3 Retrieval básico

`as_retriever(similarity_top_k=k)` nos da el buscador. Inspeccionemos los **nodos recuperados y sus scores** — es el equivalente exacto de nuestro `buscar()` casero.

In [ ]:
# ═══ 6.3 Retrieval e inspeccion de nodos ═══
recuperador = indice.as_retriever(similarity_top_k=3)

def mostrar_nodos(nodos):
    for n in nodos:
        print(f"[score {n.score:.3f}] {n.metadata.get('file_name')} (idioma: {n.metadata.get('idioma')})")
        contenido = " ".join(n.get_content().split())
        print(f"   {contenido}\n")

nodos = recuperador.retrieve("¿Qué días hay apoyo escolar?")
mostrar_nodos(nodos)

### 6.4 Ejercicio: 5 preguntas, evaluación a ojo

▶️ Hagamos 5 preguntas de distinto tipo y evaluemos **a ojo** si los chunks recuperados sirven para responderlas. Este hábito — mirar lo que trae el retrieval **antes** de conectar el LLM — es la herramienta de debugging más importante de RAG.

In [ ]:
# ═══ 6.4 Cinco preguntas al indice ═══
preguntas_ejercicio = [
    "¿Cuántos libros puedo llevarme a la vez?",              # directa, en el corpus
    "Quando acontece a oficina de compostagem?",             # en portugues, doc PT
    "¿Puedo imprimir piezas en 3D en la biblioteca?",        # parafraseo, no literal
    "¿Qué decidió la asamblea sobre el horario?",            # requiere doc especifico
    "¿Cuál es el presupuesto anual de la biblioteca?",       # NO esta en el corpus
]

for pregunta in preguntas_ejercicio:
    print("─" * 70)
    print(f"❓ {pregunta}\n")
    mostrar_nodos(recuperador.retrieve(pregunta))

print("─" * 70)
print("""📝 Que mirar:
  - la ultima pregunta NO tiene respuesta en el corpus... pero el retriever
    igual devuelve 'algo' (los k vecinos mas cercanos SIEMPRE existen).
    Detectar ese caso es trabajo del LLM + prompt (secciones 8 y 10)""")

### 6.5 Persistencia simple

El índice in-memory vive en la RAM: si reiniciás el kernel, hay que re-embeddear todo. `StorageContext` permite **guardarlo en disco y recargarlo**.

In [ ]:
# ═══ 6.5 Guardar y recargar el indice ═══
from llama_index.core import StorageContext, load_index_from_storage

CARPETA_INDICE = "indice_guardado"

indice.storage_context.persist(persist_dir=CARPETA_INDICE)
archivos_guardados = os.listdir(CARPETA_INDICE)
print(f"💾 Índice guardado en '{CARPETA_INDICE}/': {archivos_guardados}")

# Recargar (asi arrancaria una sesion nueva, sin re-embeddear el corpus)
contexto_almacenamiento = StorageContext.from_defaults(persist_dir=CARPETA_INDICE)
indice_recargado = load_index_from_storage(contexto_almacenamiento)

nodos = indice_recargado.as_retriever(similarity_top_k=1).retrieve("horario de la biblioteca")
print(f"\n✅ Índice recargado y funcionando: [{nodos[0].score:.3f}] {nodos[0].metadata.get('file_name')}")

### 6.6 Cuándo alcanza con in-memory

- ✅ Corpus chico (cientos o pocos miles de chunks)
- ✅ Prototipo / experimentación / este taller
- ✅ Un solo proceso accediendo al índice
- ❌ Corpus grande, múltiples usuarios, filtros complejos, actualizaciones frecuentes → **sección 7 (Qdrant)**

🟢 Si estás en el pathway de exploración simple, podés saltar directo a la **sección 8**.

---
## 7. Qdrant: base de datos vectorial

### 7.1 Por qué una base vectorial

El índice in-memory recalcula y recarga todo en RAM. Una **base de datos vectorial** como Qdrant ofrece:

- **Escala**: millones de vectores con búsqueda aproximada en milisegundos.
- **Persistencia real**: los vectores viven en disco, con transacciones.
- **Filtros**: "buscá solo en documentos de 2024", "solo en portugués".
- **Múltiples colecciones**: un corpus por proyecto, misma base.

### 7.2 Setup: modo embebido

`QdrantClient(path="...")` crea una base **local embebida** — un directorio en disco, sin servidor ni Docker. Para producción con múltiples procesos se usa el servidor, pero la API es la misma.

### 7.3 Conceptos

| Concepto Qdrant | Equivalente en lo que ya hicimos |
|---|---|
| **Colección** | nuestro índice (con su métrica de distancia: coseno, claro) |
| **Punto** | un chunk embeddeado (vector + id) |
| **Payload** | la metadata que viaja con el chunk (`file_name`, `idioma`) |

In [ ]:
# ═══ 7.2-7.4 Qdrant embebido como backend del mismo VectorStoreIndex ═══
from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore

RUTA_QDRANT = "qdrant_datos"
NOMBRE_COLECCION = "biblioteca_arroyo_claro"

cliente_qdrant = QdrantClient(path=RUTA_QDRANT)  # base embebida: un directorio en disco

# Si re-ejecutamos el notebook, borramos la coleccion para no duplicar puntos
if cliente_qdrant.collection_exists(NOMBRE_COLECCION):
    cliente_qdrant.delete_collection(NOMBRE_COLECCION)

almacen_vectorial = QdrantVectorStore(client=cliente_qdrant, collection_name=NOMBRE_COLECCION)
contexto_qdrant = StorageContext.from_defaults(vector_store=almacen_vectorial)

# Mismo codigo que la seccion 6 — solo cambia el storage_context. Esa es la gracia:
# migrar de in-memory a base vectorial sin tocar el resto del pipeline.
t0 = time.time()
indice_qdrant = VectorStoreIndex.from_documents(
    documentos, storage_context=contexto_qdrant, show_progress=True
)
print(f"✅ Corpus indexado en Qdrant en {time.time() - t0:.1f}s")

info = cliente_qdrant.get_collection(NOMBRE_COLECCION)
config_vectores = info.config.params.vectors
if isinstance(config_vectores, dict):  # colecciones con vectores "nombrados"
    config_vectores = list(config_vectores.values())[0]
print(f"   Colección '{NOMBRE_COLECCION}': {info.points_count} puntos")
print(f"   Métrica de distancia: {config_vectores.distance}")
print(f"   Guardada en disco en: {RUTA_QDRANT}/")

### 7.5 Filtros por metadata

Acá aparece el primer superpoder que el índice in-memory no tenía: **buscar solo dentro de un subconjunto** definido por metadata — una fuente, una fecha, una categoría... o un idioma.

In [ ]:
# ═══ 7.5 Retrieval con filtro por metadata ═══
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator

pregunta_filtro = "¿qué actividades hay en el patio?"

# Sin filtro
print("SIN filtro:")
mostrar_nodos(indice_qdrant.as_retriever(similarity_top_k=2).retrieve(pregunta_filtro))

# Con filtro: SOLO documentos en portugues
filtro_pt = MetadataFilters(filters=[
    MetadataFilter(key="idioma", value="pt", operator=FilterOperator.EQ)
])
print("CON filtro idioma == 'pt':")
recuperador_pt = indice_qdrant.as_retriever(similarity_top_k=2, filters=filtro_pt)
mostrar_nodos(recuperador_pt.retrieve(pregunta_filtro))

# 🧪 Proba esto: filtra por file_name == 'acta_asamblea_2024.md' y pregunta
# por el horario: garantiza que la respuesta salga del acta y no de otro documento

### 7.6 Ejercicio: comparar Qdrant vs in-memory

Mismo corpus, mismos embeddings, misma métrica → deberían recuperar (casi) lo mismo. Verifiquémoslo.

In [ ]:
# ═══ 7.6 In-memory vs Qdrant: ¿recuperan lo mismo? ═══
preguntas_comparacion = [
    "¿cuántas computadoras tiene la sala digital?",
    "¿cómo funciona el intercambio de semillas?",
    "quais são as regras de empréstimo?",
]

for pregunta in preguntas_comparacion:
    nodos_memoria = indice.as_retriever(similarity_top_k=2).retrieve(pregunta)
    nodos_qdrant = indice_qdrant.as_retriever(similarity_top_k=2).retrieve(pregunta)
    print(f"❓ {pregunta}")
    for etiqueta, nodos in [("in-memory", nodos_memoria), ("qdrant   ", nodos_qdrant)]:
        fuentes = [f"{n.metadata.get('file_name')} ({n.score:.3f})" for n in nodos]
        print(f"   {etiqueta} → {fuentes}")
    print()

print("📝 Los scores pueden diferir en decimales (implementaciones distintas del")
print("   mismo calculo), pero las fuentes recuperadas deberian coincidir.")

---
## 8. Pipeline RAG completo

Llegó el momento de conectar la última pieza: el **LLM generador**. Retrieval + prompt + generación = RAG completo.

### 8.1 Conectar el LLM local

Cargamos el modelo elegido en la sección 1.4 (`Qwen3-1.7B` o `Qwen3-0.6B`) vía Hugging Face y lo conectamos a LlamaIndex con la integración `HuggingFaceLLM`. Todo corre **local**: ninguna pregunta ni documento sale de tu máquina.

In [ ]:
# ═══ 8.1a Cargar el LLM generador ═══
from transformers import AutoTokenizer, AutoModelForCausalLM

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # silenciar avisos informativos de generacion

print(f"⏳ Cargando LLM: {nombre_llm}")
print("   (la primera vez descarga los pesos, puede tardar varios minutos)")

tokenizador = AutoTokenizer.from_pretrained(nombre_llm)
modelo_llm = AutoModelForCausalLM.from_pretrained(nombre_llm, dtype="auto")
modelo_llm = modelo_llm.to(dispositivo).eval()
n_parametros = sum(p.numel() for p in modelo_llm.parameters())
print(f"✅ LLM cargado: {n_parametros/1e9:.1f}B parámetros en '{dispositivo}'")

In [ ]:
# ═══ 8.1b Conectar el LLM a LlamaIndex ═══
from llama_index.llms.huggingface import HuggingFaceLLM

SYSTEM_PROMPT_RAG = (
    "Respondés preguntas sobre los documentos de la Biblioteca Comunitaria Arroyo Claro. "
    "Respondé de forma breve y precisa, únicamente con la información del contexto dado. "
    "Si la información no está en el contexto, decí que no la encontraste en los documentos."
)

def prompt_de_chat(prompt_texto):
    """Convierte el prompt plano que arma LlamaIndex al formato de chat del modelo
    (con nuestro system prompt, y sin modo razonamiento)."""
    mensajes = [
        {"role": "system", "content": SYSTEM_PROMPT_RAG},
        {"role": "user", "content": prompt_texto},
    ]
    return tokenizador.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

Settings.llm = HuggingFaceLLM(
    model=modelo_llm,                  # el modelo que ya cargamos en 8.1a
    tokenizer=tokenizador,
    max_new_tokens=256,                # respuestas cortas y al punto
    completion_to_prompt=prompt_de_chat,
    generate_kwargs={"do_sample": False,  # determinístico: equivale a temperature=0
                     "pad_token_id": tokenizador.eos_token_id},
)
print("✅ LLM conectado a LlamaIndex (Settings.llm)")
print("   - do_sample=False → generación determinística (equivale a temperature=0)")
print("   - max_new_tokens=256 → respuestas cortas y al punto")

### 8.2 Query engine: RAG en una línea

`as_query_engine()` junta todo: **retrieval** (top-k chunks) + **prompt** (plantilla con el contexto inyectado) + **generación**.

In [ ]:
# ═══ 8.2 Primer pipeline RAG completo ═══
motor_consultas = indice.as_query_engine(similarity_top_k=3)

t0 = time.time()
respuesta = motor_consultas.query("¿Cuántos libros me puedo llevar y por cuánto tiempo?")
print(f"🤖 RESPUESTA ({time.time() - t0:.1f}s):\n{respuesta}\n")

print("📚 FUENTES USADAS:")
for nodo in respuesta.source_nodes:
    print(f"   [{nodo.score:.3f}] {nodo.metadata.get('file_name')}")

### 8.3 Abrir la caja: ¿qué recibe realmente el LLM?

Nada de magia: el query engine usa una **plantilla de prompt** y le inyecta los chunks recuperados. Veámosla, y veamos también el prompt final concreto de la última consulta.

In [ ]:
# ═══ 8.3a La plantilla que usa el query engine ═══
plantillas = motor_consultas.get_prompts()
print("Plantillas del query engine:", list(plantillas.keys()), "\n")
print("═══ text_qa_template (la principal) ═══\n")
print(plantillas["response_synthesizer:text_qa_template"].get_template())

In [ ]:
# ═══ 8.3b El prompt final, reconstruido a mano ═══
pregunta_demo = "¿Cuántos libros me puedo llevar y por cuánto tiempo?"
nodos_demo = indice.as_retriever(similarity_top_k=3).retrieve(pregunta_demo)
contexto_demo = "\n\n".join(n.get_content() for n in nodos_demo)

prompt_final = plantillas["response_synthesizer:text_qa_template"].format(
    context_str=contexto_demo, query_str=pregunta_demo
)
print("Esto es (esencialmente) lo que recibe el LLM:\n")
print("┌" + "─" * 68)
for linea in prompt_final.split("\n"):
    print("│ " + linea)
print("└" + "─" * 68)

**`response_mode`: qué hacer cuando el contexto no entra en un prompt**

| Modo | Estrategia | Cuándo |
|---|---|---|
| `compact` (default) | concatena todos los chunks en la menor cantidad de prompts posible | corpus chicos, el nuestro |
| `refine` | responde con el primer chunk y *refina* la respuesta chunk a chunk | muchos chunks, respuestas largas |
| `tree_summarize` | resume por niveles, como un torneo | resúmenes de corpus enteros |

Se elige con `index.as_query_engine(response_mode="refine")`.

### 8.4 Parámetros que importan

- **`similarity_top_k`**: cuántos chunks entran al prompt. Más no siempre es mejor: contexto irrelevante distrae al modelo.
- **System prompt**: "respondé solo con el contexto dado" — ya lo configuramos en 8.1b, y es la primera defensa contra alucinaciones.
- **Temperature baja / determinística** (`do_sample=False`): para respuestas factuales no queremos creatividad.

### 8.5 Primer ejercicio: preguntas dentro y fuera del corpus

La prueba de fuego de un pipeline RAG honesto: ¿qué hace cuando la respuesta **no está** en los documentos?

In [ ]:
# ═══ 8.5 Dentro vs fuera del corpus ═══
preguntas_prueba = [
    ("EN el corpus",    "¿Qué variedades de semillas tiene el banco de semillas?"),
    ("EN el corpus",    "Quando são recebidas doações de livros em português?"),
    ("FUERA del corpus", "¿Cuál es el presupuesto anual de la biblioteca?"),
    ("FUERA del corpus", "¿Quién es la persona que preside la comisión directiva?"),
]

for categoria, pregunta in preguntas_prueba:
    print("═" * 70)
    print(f"[{categoria}] ❓ {pregunta}")
    respuesta = motor_consultas.query(pregunta)
    print(f"🤖 {respuesta}")
    fuentes = {n.metadata.get("file_name") for n in respuesta.source_nodes}
    print(f"📚 fuentes: {fuentes}")

print("═" * 70)
print("""📝 Que mirar:
  - en las preguntas FUERA del corpus, el retrieval igual trae chunks (los
    vecinos mas cercanos siempre existen) — el modelo deberia decir que no
    encontro la informacion. ¿Lo hizo? ¿O 'completo' con algo inventado?
  - este comportamiento depende del system prompt Y del tamaño del modelo:
    los modelos chicos se abstienen peor""")

### 8.6 Ir más allá: plantilla de prompt en español

La plantilla default de LlamaIndex está **en inglés**. El modelo la entiende, pero mezclar idiomas en el prompt puede degradar respuestas — y queremos control total sobre la instrucción. Reemplacémosla por una versión en español que además pida **abstención explícita**.

In [ ]:
# ═══ 8.6 Plantilla en español con abstencion explicita ═══
from llama_index.core import PromptTemplate

plantilla_es = PromptTemplate(
    "La información de contexto está a continuación.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Usando únicamente la información del contexto (y no tu conocimiento previo), "
    "respondé la pregunta en el mismo idioma en que fue formulada.\n"
    "Si la respuesta no está en el contexto, respondé exactamente: "
    "'No encontré esta información en los documentos.'\n"
    "Pregunta: {query_str}\n"
    "Respuesta: "
)

motor_consultas.update_prompts({"response_synthesizer:text_qa_template": plantilla_es})
print("✅ Plantilla reemplazada. Repetimos las preguntas problematicas:\n")

for pregunta in ["¿Cuál es el presupuesto anual de la biblioteca?",
                 "¿Qué variedades de semillas tiene el banco de semillas?"]:
    print(f"❓ {pregunta}")
    print(f"🤖 {motor_consultas.query(pregunta)}\n")

# 🧪 Proba esto: traduci la plantilla al portugues y proba con preguntas en PT

---
## 9. Reranking: refinar lo recuperado

### 9.1 El problema del top-k

El retrieval por embeddings es **rápido pero grosero**: comprime cada texto en un solo vector *antes* de conocer la pregunta. Lo más *similar* no siempre es lo más *relevante* (lo discutimos en 4.5).

### 9.2 Bi-encoder vs cross-encoder

```
  BI-ENCODER (embeddings, secciones 3-8)      CROSS-ENCODER (reranker)

  pregunta ──→ [vector]  ┐                    ┌──────────────────────┐
                         ├─→ coseno           │  pregunta + chunk    │ ──→ score
  chunk ─────→ [vector]  ┘                    │  (leidos JUNTOS)     │
                                              └──────────────────────┘
  ✔ rapidisimo: vectores precalculados        ✔ mucho mas preciso: ve la
  ✘ cada texto se comprime "a ciegas",          interaccion pregunta-chunk
    sin conocer la pregunta                   ✘ caro: un forward pass por
                                                cada par (pregunta, chunk)
```

La estrategia estándar es un **embudo**: el bi-encoder filtra millones → top-20 baratos; el cross-encoder reordena esos 20 → top-5 buenos. Lo caro se aplica solo sobre lo poco.

### 9.3 Implementación

`Qwen3-Reranker-0.6B` recibe el par (pregunta, chunk) y devuelve la probabilidad de que el chunk sea relevante. Lo integramos como **node postprocessor** de LlamaIndex: un paso que transforma los nodos recuperados antes de la generación.

In [ ]:
# ═══ 9.3a Cargar el reranker ═══
print(f"⏳ Cargando reranker: {NOMBRE_RERANKER}")
tokenizador_rr = AutoTokenizer.from_pretrained(NOMBRE_RERANKER, padding_side="left")
modelo_rr = AutoModelForCausalLM.from_pretrained(NOMBRE_RERANKER, dtype="auto").to(dispositivo).eval()

# El reranker de Qwen es un LLM que responde "yes"/"no" a la pregunta
# "¿este documento responde esta consulta?" — el score es P("yes")
ID_SI = tokenizador_rr.convert_tokens_to_ids("yes")
ID_NO = tokenizador_rr.convert_tokens_to_ids("no")

PREFIJO_RR = (
    "<|im_start|>system\nJudge whether the Document meets the requirements based on "
    "the Query and the Instruct provided. Note that the answer can only be \"yes\" or "
    "\"no\".<|im_end|>\n<|im_start|>user\n"
)
SUFIJO_RR = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
INSTRUCCION_RR = "Given a web search query, retrieve relevant passages that answer the query"

def puntuar_relevancia(pregunta, texto_chunk):
    """Devuelve P(relevante) segun el cross-encoder, entre 0 y 1."""
    entrada = (f"{PREFIJO_RR}<Instruct>: {INSTRUCCION_RR}\n"
               f"<Query>: {pregunta}\n<Document>: {texto_chunk}{SUFIJO_RR}")
    tokens = tokenizador_rr(entrada, return_tensors="pt", truncation=True,
                            max_length=2048).to(dispositivo)
    with torch.no_grad():
        logits = modelo_rr(**tokens).logits[0, -1]
    return torch.softmax(torch.stack([logits[ID_NO], logits[ID_SI]]), dim=0)[1].item()

# Prueba rapida
p = "¿cuántos libros puedo llevarme?"
print(f"\n'{p}' vs chunk del reglamento:  {puntuar_relevancia(p, 'Cada persona puede llevarse hasta 3 libros a la vez, por 21 días.'):.3f}")
print(f"'{p}' vs chunk de compostaje:   {puntuar_relevancia(p, 'A oficina de compostagem acontece na última terça-feira do mês.'):.3f}")

In [ ]:
# ═══ 9.3b Integrarlo como node postprocessor de LlamaIndex ═══
from typing import List, Optional
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.schema import NodeWithScore, QueryBundle

class RerankerQwen(BaseNodePostprocessor):
    """Reordena los nodos con el cross-encoder, filtra por umbral y devuelve top_n."""
    top_n: int = 3
    umbral: float = 0.5   # P(relevante) minima para entrar al contexto

    def _postprocess_nodes(
        self, nodes: List[NodeWithScore], query_bundle: Optional[QueryBundle] = None
    ) -> List[NodeWithScore]:
        for nodo in nodes:
            nodo.score = puntuar_relevancia(query_bundle.query_str, nodo.get_content())
        ordenados = sorted(nodes, key=lambda n: n.score, reverse=True)
        filtrados = [n for n in ordenados if n.score >= self.umbral]
        return (filtrados or ordenados[:1])[: self.top_n]  # nunca dejar el contexto vacio

# El embudo: recuperar 10 baratos con embeddings, y quedarnos solo con los que el
# cross-encoder considera realmente relevantes (hasta 3)
motor_con_reranker = indice.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[RerankerQwen(top_n=3, umbral=0.5)],
)
motor_con_reranker.update_prompts({"response_synthesizer:text_qa_template": plantilla_es})
print("✅ Query engine con reranking: top-10 por embeddings → cross-encoder → umbral 0.5")
print("   Detalle clave: los scores del cross-encoder son probabilidades, asi que")
print("   PODEMOS filtrar por umbral. Con la similaridad coseno no hay umbral universal:")
print("   0.3 puede ser 'muy relevante' en un corpus y ruido en otro.")

### 9.4 Ejercicio: con y sin reranking, misma pregunta

Usemos una pregunta con **distractores**: *"¿Qué materiales no se prestan a domicilio?"*. La respuesta está en el reglamento (los materiales de referencia no salen de la sala de lectura), pero el documento de compostaje también lista "lo que no puede ir" — un vecino temático que **suena** parecido sin responder la pregunta.

Comparemos qué chunks llegan al LLM, **con qué scores**, y cuánto cuesta en latencia.

In [ ]:
# ═══ 9.4 Comparacion con/sin reranking ═══
pregunta_dificil = "¿Qué materiales no se prestan a domicilio?"

# Sin reranking: top-3 directo por embeddings
t0 = time.time()
respuesta_sin = motor_consultas.query(pregunta_dificil)
lat_sin = time.time() - t0

# Con reranking: top-10 → cross-encoder → top-3
t0 = time.time()
respuesta_con = motor_con_reranker.query(pregunta_dificil)
lat_con = time.time() - t0

print(f"❓ {pregunta_dificil}\n")
print(f"── SIN reranking ({lat_sin:.1f}s) — scores = similaridad coseno:")
for n in respuesta_sin.source_nodes:
    print(f"   [{n.score:.3f}] {n.metadata.get('file_name')}")
print(f"🤖 {respuesta_sin}\n")
print(f"── CON reranking ({lat_con:.1f}s) — scores = P(relevante) del cross-encoder:")
for n in respuesta_con.source_nodes:
    print(f"   [{n.score:.3f}] {n.metadata.get('file_name')}")
print(f"🤖 {respuesta_con}\n")

print("📝 Que paso aca:")
print("   - por coseno los tres chunks parecen igual de 'cercanos' (~0.2-0.3), asi")
print("     que el contexto se llena de vecinos tematicos que NO responden — y ese")
print("     ruido puede confundir al modelo (el 'mas no siempre es mejor' de 8.4)")
print("   - el cross-encoder, que lee pregunta y chunk JUNTOS, es categorico:")
print("     reglamento con score altisimo, distractores casi en cero → el umbral")
print("     los deja afuera y al LLM le llega un contexto chico pero limpio")

In [ ]:
# ═══ 9.4b Visualizar el costo en latencia ═══
fig = go.Figure(go.Bar(
    x=["sin reranking", "con reranking"],
    y=[lat_sin, lat_con],
    marker=dict(color=COLORES[0], cornerradius=4), width=0.5,
    text=[f"{lat_sin:.1f}s", f"{lat_con:.1f}s"], textposition="outside",
    hovertemplate="%{x}: %{y:.2f}s<extra></extra>",
))
estilizar_figura(fig, "Latencia de la consulta completa (retrieval + generación)")
fig.update_layout(yaxis_title="segundos", height=380)
fig.show()

print("📝 El sobrecosto del reranking = 10 forward passes del cross-encoder.")
print("   En corpus chicos puede no cambiar mucho la respuesta; su valor aparece")
print("   con corpus grandes, preguntas ambiguas y cuando el error cuesta caro.")

### 9.5 Discusión: ¿cuándo justifica el costo?

- **Corpus grande**: con 100 chunks, el top-3 por embeddings suele alcanzar; con 100.000, el embudo top-50 → top-5 cambia todo.
- **Preguntas ambiguas**: negaciones, condiciones, preguntas que mezclan temas.
- **Alto costo del error**: normativas, salud, decisiones — vale pagar segundos extra por precisión.
- **Presupuesto de contexto**: con un LLM chico, mandar 3 chunks *buenos* rinde más que mandar 10 mediocres.

---
## 10. Citaciones y reducción de alucinaciones

### 10.1 Alucinaciones en RAG

RAG **reduce** las alucinaciones (el modelo tiene el contexto correcto a mano) pero **no las elimina**: el modelo puede ignorar el contexto, distorsionarlo o rellenar huecos. Tipos frecuentes:

- **Respuesta sin sustento**: afirma algo que no está en ninguna fuente recuperada.
- **Mezcla de fuentes**: combina datos de dos documentos en una afirmación falsa (p. ej. el horario del taller de huerta con el día del club de lectura).
- **Sobreconfianza**: el contexto no alcanza, pero responde igual, con total seguridad.

### 10.2 Grounding: atar la respuesta a las fuentes

**Grounding** = exigir que cada afirmación salga del contexto recuperado. Las dos herramientas básicas ya las usamos: system prompt restrictivo y **abstención explícita** ("No encontré esta información en los documentos", sección 8.6). La tercera es estructural: **citaciones**.

### 10.3 Citaciones como mecanismo

El `CitationQueryEngine` de LlamaIndex numera cada chunk como fuente `[1]`, `[2]`, ... y le pide al modelo citar cada afirmación:

- **La cita es la unidad verificable**: quien lee puede ir a la fuente `[2]` y chequear.
- **Citar cambia el comportamiento del modelo**: al tener que atribuir cada frase, le cuesta más inventar. La estructura del prompt empuja al grounding.

In [ ]:
# ═══ 10.3 CitationQueryEngine con plantilla en español ═══
from llama_index.core.query_engine import CitationQueryEngine

plantilla_citas_es = PromptTemplate(
    "A continuación hay fuentes numeradas. Usá únicamente esas fuentes para responder.\n"
    "Respondé con oraciones completas y agregá el número de fuente después de cada "
    "afirmación, por ejemplo [1].\n"
    "Por ejemplo:\n"
    "Source 1:\nEl cielo puede verse rojizo al atardecer.\n"
    "Source 2:\nEl agua de lluvia moja las calles.\n"
    "Pregunta: ¿De qué color se ve el cielo al atardecer?\n"
    "Respuesta: Al atardecer el cielo puede verse rojizo [1].\n"
    "Si la respuesta no está en las fuentes, respondé: "
    "'No encontré esta información en los documentos.'\n"
    "Ahora las fuentes reales:\n"
    "------\n"
    "{context_str}\n"
    "------\n"
    "Pregunta: {query_str}\n"
    "Respuesta: "
)

motor_citas = CitationQueryEngine.from_args(
    indice,
    similarity_top_k=3,
    citation_chunk_size=256,            # re-parte los chunks en fuentes citables mas finas
    citation_qa_template=plantilla_citas_es,
)
print("✅ Query engine con citaciones listo")

### 10.4 Ejercicio: misma pregunta, con y sin citaciones

Y el paso clave, que ninguna herramienta hace por nosotros: **verificar manualmente** si cada cita realmente respalda la afirmación.

In [ ]:
# ═══ 10.4 Query engine normal vs con citaciones ═══
pregunta_citas = "¿Qué decidió la asamblea de 2024 sobre el horario y la comisión?"

print(f"❓ {pregunta_citas}\n")
print("── SIN citaciones:")
print(f"🤖 {motor_consultas.query(pregunta_citas)}\n")

print("── CON citaciones:")
respuesta_citada = motor_citas.query(pregunta_citas)
print(f"🤖 {respuesta_citada}\n")

print("📚 FUENTES NUMERADAS (para verificar a mano):")
for i, nodo in enumerate(respuesta_citada.source_nodes, start=1):
    contenido = " ".join(nodo.get_content().split())
    print(f"\n[{i}] ({nodo.metadata.get('file_name')})")
    print(f"    {contenido[:220]}{'...' if len(contenido) > 220 else ''}")

In [ ]:
# ═══ 10.4b Verificacion manual guiada ═══
print("""🔎 EJERCICIO DE VERIFICACION (a mano, en grupo):

  1. Tomá cada afirmacion de la respuesta citada de arriba.
  2. Anda a la fuente [n] que la acompaña y verifica:
       ¿La fuente realmente dice eso?           → cita CORRECTA
       ¿La fuente habla de otra cosa?           → cita MAL ATRIBUIDA
       ¿La afirmacion no aparece en NINGUNA?    → cita INVENTADA
  3. Prestale especial atencion a los numeros (horarios, cantidades, años):
     son el lugar clasico donde el modelo mezcla fuentes.

  🧪 Proba esto: hace una pregunta cuya respuesta combine DOS documentos
  ("¿que actividades hay los martes y que se decidio sobre el horario?")
  y verifica si cada mitad de la respuesta cita el documento correcto.""")

### 10.5 Estrategias complementarias

Las citaciones no reemplazan al resto del arsenal — se suman:

- **System prompt restrictivo** ("respondé solo con el contexto") — sección 8.4.
- **Generación determinística** (`do_sample=False` / `temperature=0`) para respuestas factuales — sección 8.1b.
- **Mostrar siempre los chunks fuente junto a la respuesta** en la interfaz: aunque el modelo no cite, quien usa el sistema puede verificar. Es la más simple y la más robusta.

### 10.6 Discusión: trazabilidad como práctica de cuidado

- ¿**Quién verifica** las respuestas antes de que lleguen a otras personas? ¿El sistema muestra las fuentes para que verificar sea posible?
- ¿**Quién responde por el error** si el sistema da información equivocada (un horario, un requisito, un derecho)?
- ¿**Qué error sería más grave** en tu proyecto: inventar información, u omitir información que sí estaba?

Para sistemas comunitarios, la respuesta citada no es un lujo técnico: es lo que permite que la confianza no dependa de "creerle a la máquina".

---
## 11. Evaluación de RAG

### 11.1 Dos cosas distintas a evaluar

Un pipeline RAG puede fallar en dos lugares distintos, y conviene medirlos por separado:

1. **Retrieval**: ¿recuperamos los chunks correctos? (si acá falla, todo lo demás falla)
2. **Generación**: dado un buen contexto, ¿la respuesta es **fiel** al contexto y **responde** la pregunta?

### 11.2 Evaluar retrieval: hit rate

Armamos un **mini set de evaluación**: preguntas + el documento donde está su respuesta. La métrica más simple es el **hit rate@k**: ¿apareció el documento correcto entre los k chunks recuperados?

Un detalle importante: incluimos preguntas **directas** (usan palabras del documento) y preguntas **difíciles** (parafraseo, formulación indirecta, o preguntar en español algo que está en un documento en portugués). Si el set solo tiene preguntas fáciles, todo da 100% y la evaluación no discrimina entre configuraciones.

In [ ]:
# ═══ 11.2a Mini set de evaluacion: pregunta → documento esperado ═══
# Mitad directas, mitad dificiles (parafraseo, pregunta indirecta, cruce de
# idiomas): si todas las preguntas son faciles, la metrica da 100% y no
# aprendemos nada del pipeline.
set_evaluacion = [
    # directas
    ("¿Qué día es el taller de ajedrez?",                     "talleres_y_actividades.md"),
    ("¿Cómo reservo la impresora 3D?",                        "equipamiento_sala_digital.md"),
    ("O que não pode ir na composteira?",                     "oficina_compostagem_pt.md"),
    ("¿Qué variedades de semillas hay disponibles?",          "guia_huerta_semillas.md"),
    # dificiles
    ("¿Qué pasa si devuelvo un libro tarde?",                 "reglamento_prestamos.md"),
    ("¿Dónde funcionaba la biblioteca al principio?",         "historia_biblioteca.md"),
    ("¿Puedo donar una colección grande de libros?",          "acervo_emprestimos_pt.md"),
    ("¿Qué van a hacer con la plata de la feria del libro?",  "acta_asamblea_2024.md"),
]

def hit_rate(indice_eval, k):
    """Proporcion de preguntas cuyo documento esperado aparece en el top-k."""
    aciertos = 0
    recuperador_eval = indice_eval.as_retriever(similarity_top_k=k)
    for pregunta, doc_esperado in set_evaluacion:
        fuentes = [n.metadata.get("file_name") for n in recuperador_eval.retrieve(pregunta)]
        aciertos += doc_esperado in fuentes
    return aciertos / len(set_evaluacion)

for k in [1, 3, 5]:
    tasa = hit_rate(indice, k)
    print(f"hit rate@{k}: {tasa:.0%}  {'█' * int(tasa * 30)}")

In [ ]:
# ═══ 11.2b El efecto de chunk_size y top_k sobre el retrieval ═══
print("⏳ Re-indexando el corpus con dos chunk_size distintos (corpus chico, es rapido)...")

indices_por_chunk = {}
for tamanio in [96, 512]:
    splitter = SentenceSplitter(chunk_size=tamanio, chunk_overlap=20)
    nodos_eval = splitter.get_nodes_from_documents(documentos)
    indices_por_chunk[tamanio] = VectorStoreIndex(nodos_eval)

valores_k = [1, 2, 3, 5]
fig = go.Figure()
for (tamanio, indice_eval), color in zip(indices_por_chunk.items(), COLORES):
    tasas = [hit_rate(indice_eval, k) for k in valores_k]
    fig.add_trace(go.Scatter(
        x=valores_k, y=tasas, mode="lines+markers",
        name=f"chunk_size={tamanio}",
        line=dict(color=color, width=2), marker=dict(size=9),
        hovertemplate="top_k=%{x}: %{y:.0%}<extra>chunk_size=" + str(tamanio) + "</extra>",
    ))

estilizar_figura(fig, "Hit rate del retrieval según top_k y chunk_size")
fig.update_layout(xaxis_title="similarity_top_k", yaxis_title="hit rate",
                  yaxis_tickformat=".0%", yaxis_range=[0, 1.05], height=420)
fig.update_xaxes(tickvals=valores_k)
fig.show()

print("📝 Subir top_k siempre ayuda al hit rate... pero mete mas contexto (y mas")
print("   ruido) en el prompt. El chunk_size optimo depende del corpus: por eso")
print("   se mide en lugar de adivinarse.")

### 11.3 Evaluar generación: faithfulness y relevancia

- **Faithfulness (fidelidad)**: ¿cada afirmación de la respuesta se sostiene en el contexto recuperado?
- **Relevancia**: ¿la respuesta responde *lo que se preguntó*?

La forma industrial de medirlas es **LLM-as-judge** (otro LLM más grande evalúa cada respuesta — LlamaIndex trae `FaithfulnessEvaluator` y `RelevancyEvaluator`). Con modelos chicos y locales como los nuestros, el juez no es confiable; una verificación simple y honesta a nuestra escala: chequear que los **datos duros** de la respuesta (números, días, años) aparezcan en el contexto usado.

In [ ]:
# ═══ 11.3 Verificacion simple de fidelidad: los datos duros de la respuesta,
#     ¿aparecen en el contexto que uso el modelo? ═══
def datos_duros(texto):
    """Extrae numeros y cifras de un texto (los candidatos clasicos a alucinacion)."""
    return set(re.findall(r"\d+(?:[:.]\d+)?", texto))

preguntas_generacion = [
    "¿Hasta cuántas horas por semana puedo reservar la impresora 3D?",
    "¿Cuántas personas integran la comisión directiva después de la asamblea de 2024?",
]

for pregunta in preguntas_generacion:
    respuesta = motor_consultas.query(pregunta)
    contexto_usado = " ".join(n.get_content() for n in respuesta.source_nodes)
    sin_sustento = datos_duros(str(respuesta)) - datos_duros(contexto_usado)
    print(f"❓ {pregunta}")
    print(f"🤖 {respuesta}")
    if sin_sustento:
        print(f"⚠️ Datos en la respuesta que NO aparecen en el contexto: {sin_sustento}")
        print("   (candidatos a alucinacion → verificar a mano)")
    else:
        print("✅ Todos los numeros de la respuesta estan en el contexto recuperado")
    print()

print("📝 Este chequeo es deliberadamente simple: atrapa numeros inventados, no")
print("   distorsiones sutiles. Para eso: verificacion humana (10.4) o LLM-as-judge")
print("   con un modelo mas grande (🚀 llama_index.core.evaluation)")

### 11.4 Evaluación cualitativa por equipo

Las métricas automáticas no capturan lo que más importa en un sistema comunitario. Rúbrica sugerida para evaluar en equipo (escala 1–4 por dimensión):

| Dimensión | Pregunta guía |
|---|---|
| **Utilidad** | ¿La respuesta le sirve a la persona que preguntó? |
| **Claridad** | ¿Se entiende sin conocimiento técnico? |
| **Tono** | ¿Es apropiado para la comunidad que va a usar el sistema? |
| **Sesgos** | ¿Responde igual de bien para todos los temas/idiomas del corpus? |
| **Honestidad** | ¿Dice "no sé" cuando corresponde, o inventa? |

### 11.5 Ejercicio integrador: comparar 2 configuraciones

Cerremos midiendo de punta a punta: **misma batería de preguntas, dos pipelines distintos** (chunks chicos + top_k alto vs chunks grandes + top_k bajo).

In [ ]:
# ═══ 11.5 Dos configuraciones, mismo set de preguntas ═══
config_A = {"nombre": "chunks 96 / top_k 5", "indice": indices_por_chunk[96], "k": 5}
config_B = {"nombre": "chunks 512 / top_k 2", "indice": indices_por_chunk[512], "k": 2}

print(f"{'':<42} {'hit rate':>9}")
tasas_finales = {}
for config in [config_A, config_B]:
    tasa = hit_rate(config["indice"], config["k"])
    tasas_finales[config["nombre"]] = tasa
    print(f"{config['nombre']:<42} {tasa:>8.0%}")

# Y una comparacion cualitativa de las respuestas generadas
pregunta_integradora = "¿Cuándo se fundó la biblioteca y cuántos libros tiene hoy?"
print(f"\n❓ {pregunta_integradora}\n")
for config in [config_A, config_B]:
    motor_config = config["indice"].as_query_engine(similarity_top_k=config["k"])
    motor_config.update_prompts({"response_synthesizer:text_qa_template": plantilla_es})
    t0 = time.time()
    respuesta = motor_config.query(pregunta_integradora)
    print(f"── {config['nombre']} ({time.time() - t0:.1f}s):")
    print(f"🤖 {respuesta}\n")

# 🧪 Proba esto: agrega una config con reranking (seccion 9) a la comparacion.
# ¿El hit rate mejora lo suficiente para justificar la latencia extra?

### 11.6 Discusión: la evaluación nunca está terminada

El mini set de 8 preguntas nos sirvió para el taller, pero las preguntas reales de la comunidad van a ser otras. La práctica sana:

- **Registrar las preguntas reales** (con consentimiento) y sumarlas al set de evaluación.
- **Revisar periódicamente** las respuestas con peor feedback.
- Tratar la evaluación como **monitoreo continuo**, no como un examen que se aprueba una vez.

---
## 12. Aplicación a proyectos

### 12.1 Definir el caso de uso

Antes de tocar código, tres preguntas con tu equipo:

1. **¿Qué documentos?** ¿Están digitalizados? ¿En qué formato e idioma(s)? ¿Hay información sensible que no debería entrar al corpus?
2. **¿Qué preguntas?** Escriban 10 preguntas reales que la gente hace. Ese es su primer set de evaluación (sección 11.2).
3. **¿Qué usuarios?** ¿Quién consulta? ¿Qué costo tiene una respuesta incorrecta para esa persona?

### 12.2 Elegir configuración

| Decisión | Elegí esto... | ...si |
|---|---|---|
| **Índice** | in-memory (sección 6) | prototipo, corpus chico, un solo proceso |
| | Qdrant (sección 7) | corpus creciente, filtros por metadata, persistencia |
| **Reranking** | sin (sección 8) | corpus chico, latencia importa |
| | con (sección 9) | corpus grande, preguntas ambiguas, error caro |
| **LLM** | Qwen3-1.7B | GPU o buena RAM: mejores respuestas y abstención |
| | Qwen3-0.6B | CPU / hardware limitado |

### 12.3 Implementación rápida

Todo el taller, condensado en una función. Apuntala a una carpeta con **tus** documentos y tenés tu pipeline andando.

In [ ]:
# ═══ 12.3 Pipeline minimo reutilizable ═══
def construir_pipeline_rag(carpeta_documentos, chunk_size=256, top_k=3, con_reranker=False):
    """Construye un query engine RAG completo sobre una carpeta de documentos."""
    docs = SimpleDirectoryReader(str(carpeta_documentos)).load_data()
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=int(chunk_size * 0.1))
    indice_proyecto = VectorStoreIndex(splitter.get_nodes_from_documents(docs))

    postprocesadores = [RerankerQwen(top_n=top_k)] if con_reranker else []
    k_inicial = top_k * 3 if con_reranker else top_k
    motor = indice_proyecto.as_query_engine(
        similarity_top_k=k_inicial, node_postprocessors=postprocesadores
    )
    motor.update_prompts({"response_synthesizer:text_qa_template": plantilla_es})
    print(f"✅ Pipeline listo: {len(docs)} documentos, chunk_size={chunk_size}, "
          f"top_k={top_k}, reranker={'sí' if con_reranker else 'no'}")
    return motor

# Con el corpus del taller (cambia la carpeta por 'mis_documentos' para tu proyecto):
CARPETA_PROYECTO = "mis_documentos" if os.path.isdir("mis_documentos") else CARPETA_CORPUS
motor_proyecto = construir_pipeline_rag(CARPETA_PROYECTO)

print(f"\n🤖 {motor_proyecto.query('¿Qué talleres hay y qué días?')}")

### 12.4 Próximo experimento para cada equipo

Plantilla para definir el siguiente paso (una por equipo):

> - **Corpus inicial**: [qué 5–10 documentos]
> - **10 preguntas de evaluación**: [las que la gente hace de verdad]
> - **Métrica a mirar primero**: hit rate@3 del retrieval (sección 11.2)
> - **Primera mejora a probar**: [chunking / prompt en el idioma del corpus / reranking / modelo más grande]
> - **Qué error sería más grave** y cómo lo mitigamos: [abstención / citas / revisión humana]

Regla de oro: **mejorar el retrieval antes que el generador** — es más barato y suele ser donde está el problema.

---
## 13. Cierre

### 13.1 Síntesis

Lo que construimos, pieza por pieza:

1. **Embeddings** (sección 3): texto → vector; cerca en el espacio = parecido en significado. Vimos también que el espacio no es igual de bueno para todos los idiomas.
2. **Similaridad** (sección 4): coseno en 3 líneas de numpy, y retrieval completo en 10 — antes de delegárselo a un framework.
3. **Chunking** (sección 5): no hay configuración universal; depende del documento y de las preguntas.
4. **Índice** (secciones 6-7): LlamaIndex in-memory para prototipar; Qdrant embebido para persistencia y filtros.
5. **Pipeline RAG** (sección 8): retrieval + prompt con contexto + generación local. Abrimos la caja y vimos el prompt real.
6. **Reranking** (sección 9): el embudo bi-encoder → cross-encoder, cuando la precisión lo justifica.
7. **Citaciones** (sección 10): grounding verificable; la trazabilidad como práctica de cuidado.
8. **Evaluación** (sección 11): hit rate para el retrieval, fidelidad para la generación, rúbrica cualitativa para lo que las métricas no ven.

### 13.2 Próximos pasos según pathway

- 🟢 **Exploración simple** → armá el corpus de tu proyecto (`mis_documentos/`) y usá `construir_pipeline_rag()` (sección 12.3). Después sumá citaciones (sección 10).
- 🟡 **Experimentación** → experimentá con chunking (sección 5) sobre tus documentos reales y medí hit rate con tus propias 10 preguntas (sección 11.2).
- 🔴 **Profundidad** → migrá tu índice a Qdrant con metadata útil (sección 7.5), sumá reranking (sección 9) y automatizá la evaluación (🚀 `llama_index.core.evaluation`).
- ⚙️ **Hardware limitado** → todo el pipeline funciona con `Qwen3-0.6B`; si la generación es lenta, achicá `similarity_top_k` y `max_new_tokens`, y recordá que el retrieval solo (secciones 3-7) es liviano y ya es útil por sí mismo.

**El pipeline es de ustedes: corre completo en sus máquinas, con modelos abiertos, sobre sus documentos.**